# CNN aplicada: do MLflow ao “usuário final” (com webcam)

Este notebook demonstra, de forma **prática e rastreável**, como consumir um modelo treinado na disciplina e chegar até uma experiência próxima de “produto”:

1. Selecionar um **modelo** treinado (arquivo `.pt` salvo no projeto) e/ou um artefato registrado no MLflow  
2. Carregar o modelo em modo de inferência  
3. Executar inferência em **imagens** (arquivo local e/ou URL)  
4. Executar inferência em **webcam** com exibição de *top‑k* probabilidades  

> **Observação importante (escopo):** o modelo da Aula 07 é de **classificação** (uma classe por imagem / top‑k).  
> Ele não realiza **detecção** (vários objetos no mesmo frame com bounding boxes) nem **contagem** de objetos.

---

## Pré‑requisitos

- Projeto `taia-lab` instalado (`make install` ou `pip install -e .`)
- Dependências comuns: `torch`, `torchvision`, `mlflow`, `numpy`, `opencv-python`, `Pillow`, `ipywidgets`, `matplotlib`, `requests`

Se algo falhar, instale:
```bash
pip install opencv-python ipywidgets matplotlib pillow requests
```


In [ ]:
# Imports e configuração básica
from __future__ import annotations

import io
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np
import torch
from PIL import Image

import matplotlib.pyplot as plt

# Opcional (para URL)
import requests

# Opcional (para webcam)
try:
    import cv2
except Exception as e:
    cv2 = None
    print("OpenCV (cv2) não está disponível:", e)

# Widgets (UI simples no notebook)
import ipywidgets as widgets
from IPython.display import display, clear_output


## 1) Localizar a raiz do projeto e listar modelos disponíveis

Este notebook procura automaticamente a raiz do projeto (presença de `pyproject.toml` ou `.git`) e lista arquivos em `models/*.pt`.

> Dica: isso demonstra a ideia de **artefato versionado**. O treinamento gerou um arquivo reutilizável.


In [ ]:
def find_project_root(start: Optional[Path] = None) -> Path:
    start = start or Path.cwd()
    p = start.resolve()
    for parent in [p, *p.parents]:
        if (parent / "pyproject.toml").exists() or (parent / ".git").exists():
            return parent
    # fallback: usa cwd
    return p

ROOT = find_project_root()
MODELS_DIR = ROOT / "models"
MLRUNS_DIR = ROOT / "mlruns"

print("ROOT:", ROOT)
print("MODELS_DIR:", MODELS_DIR)
print("MLRUNS_DIR:", MLRUNS_DIR)

MODELS_DIR.mkdir(parents=True, exist_ok=True)

def list_model_files(models_dir: Path):
    return sorted(models_dir.glob("*.pt"), key=lambda p: p.stat().st_mtime, reverse=True)

model_files = list_model_files(MODELS_DIR)
print("Modelos encontrados:", len(model_files))
for p in model_files[:10]:
    print(" -", p.name)


## 2) Definir classes do CIFAR‑10 e *transforms* consistentes

O pipeline de Transfer Learning (Aula 07) utilizou **MobileNetV3-Small** com entrada típica de **224×224** e normalização padrão do ImageNet.

A coerência do pré‑processamento é parte essencial de “do MLflow ao usuário final”:  
**se a transformação mudar, a inferência pode degradar mesmo com o mesmo modelo.**


In [ ]:
# Nomes das classes do CIFAR-10 (ordem padrão)
CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

# Transforms (torchvision)
import torchvision.transforms as T

IMAGE_SIZE = 224

# Normalização padrão do ImageNet (comum para backbones pré-treinados)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

TRANSFORM = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def preprocess_pil(img: Image.Image) -> torch.Tensor:
    """Retorna tensor [1, C, H, W]."""
    x = TRANSFORM(img.convert("RGB"))
    return x.unsqueeze(0)


## 3) Carregar o modelo `.pt` e criar um “predictor” simples

Assumimos que o arquivo `.pt` contém:
- `state_dict`
- metadados mínimos (ex.: `n_classes`)

Como o formato exato do `.pt` pode variar, este carregamento é robusto e falha com mensagem clara se não reconhecer o conteúdo.


In [ ]:
import torchvision

@dataclass
class LoadedModel:
    model: torch.nn.Module
    device: torch.device
    n_classes: int

def build_mobilenet_v3_small(n_classes: int) -> torch.nn.Module:
    if not hasattr(torchvision.models, "mobilenet_v3_small"):
        raise RuntimeError("torchvision não possui mobilenet_v3_small nesta versão.")
    m = torchvision.models.mobilenet_v3_small(weights=None)
    in_features = m.classifier[-1].in_features
    m.classifier[-1] = torch.nn.Linear(in_features, n_classes)
    return m

def choose_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def load_pt_model(pt_path: Path, device: Optional[torch.device] = None) -> LoadedModel:
    device = device or choose_device()
    payload = torch.load(pt_path, map_location="cpu")

    if isinstance(payload, dict) and "state_dict" in payload:
        state_dict = payload["state_dict"]
        n_classes = int(payload.get("n_classes", 10))
    elif isinstance(payload, dict):
        # Heurística: state_dict direto?
        if any(k.startswith("classifier") or k.startswith("features") for k in payload.keys()):
            state_dict = payload
            n_classes = 10
        else:
            raise ValueError(f"Formato .pt inesperado (dict sem 'state_dict'): chaves={list(payload.keys())[:10]}")
    else:
        raise ValueError(f"Formato .pt inesperado: type={type(payload)}")

    model = build_mobilenet_v3_small(n_classes=n_classes)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    model.to(device)
    model.eval()

    if missing:
        print("[AVISO] Parâmetros ausentes (strict=False):", missing[:8], "...")
    if unexpected:
        print("[AVISO] Parâmetros inesperados (strict=False):", unexpected[:8], "...")

    return LoadedModel(model=model, device=device, n_classes=n_classes)

@torch.no_grad()
def predict_topk(loaded: LoadedModel, img_pil: Image.Image, k: int = 5) -> List[Tuple[str, float]]:
    x = preprocess_pil(img_pil).to(loaded.device)
    logits = loaded.model(x)
    probs = torch.softmax(logits, dim=1).squeeze(0).detach().cpu().numpy()
    top_idx = np.argsort(-probs)[:k]
    out: List[Tuple[str, float]] = []
    for i in top_idx:
        name = CIFAR10_CLASSES[i] if i < len(CIFAR10_CLASSES) else f"class_{i}"
        out.append((name, float(probs[i])))
    return out


## 4) UI simples: selecionar modelo e testar com imagem (arquivo local ou URL)

A ideia aqui é aproximar de “produto”: você escolhe o modelo, fornece uma imagem e recebe o **top‑k** com probabilidades.


In [ ]:
# Dropdown de modelos disponíveis
model_files = list_model_files(MODELS_DIR)
options = [(p.name, str(p)) for p in model_files] if model_files else [("— nenhum modelo encontrado em models/ —", "")]

model_dropdown = widgets.Dropdown(
    options=options,
    description="Modelo:",
    layout=widgets.Layout(width="80%")
)

load_button = widgets.Button(description="Carregar modelo", button_style="primary")
status_out = widgets.Output()

loaded_model: Optional[LoadedModel] = None

def on_load_clicked(_):
    global loaded_model
    with status_out:
        clear_output()
        pt = model_dropdown.value
        if not pt:
            print("Nenhum arquivo .pt selecionado.")
            return
        pt_path = Path(pt)
        if not pt_path.exists():
            print("Arquivo não encontrado:", pt_path)
            return
        print("Carregando:", pt_path.name)
        loaded_model = load_pt_model(pt_path)
        print("OK. Device:", loaded_model.device, "n_classes:", loaded_model.n_classes)

load_button.on_click(on_load_clicked)

display(widgets.VBox([model_dropdown, load_button, status_out]))


In [ ]:
# Upload de imagem (local)
uploader = widgets.FileUpload(accept="image/*", multiple=False)
infer_button = widgets.Button(description="Inferir (imagem enviada)", button_style="success")
out_img = widgets.Output()
out_pred = widgets.Output()

def _get_upload_fileinfo(uploader_value):
    """
    Compatível com ipywidgets FileUpload em diferentes versões:
    - dict: {filename: {"content": b"...", ...}}
    - tuple/list: ({"content": b"...", ...}, ...)
    """
    if uploader_value is None:
        return None

    if isinstance(uploader_value, dict):
        if not uploader_value:
            return None
        return next(iter(uploader_value.values()))

    if isinstance(uploader_value, (tuple, list)):
        if len(uploader_value) == 0:
            return None
        return uploader_value[0]

    return None

def on_infer_upload(_):
    if loaded_model is None:
        with out_pred:
            clear_output()
            print("Carregue um modelo primeiro.")
        return

    file_info = _get_upload_fileinfo(uploader.value)
    if not file_info:
        with out_pred:
            clear_output()
            print("Envie uma imagem (upload) primeiro.")
        return

    raw = file_info["content"]
    img = Image.open(io.BytesIO(raw)).convert("RGB")

    preds = predict_topk(loaded_model, img, k=5)

    with out_img:
        clear_output()
        plt.figure(figsize=(4,4))
        plt.imshow(img)
        plt.axis("off")
        plt.show()

    with out_pred:
        clear_output()
        print("Top-5:")
        for name, p in preds:
            print(f"  {name:<12}  {p:.3f}")

infer_button.on_click(on_infer_upload)

display(widgets.VBox([widgets.HTML("<b>1) Inferência por upload</b>"), uploader, infer_button, out_img, out_pred]))

In [ ]:
# Inferência por URL (Textarea costuma colar melhor no VS Code)
url_text = widgets.Textarea(
    value="",
    placeholder="Cole a URL aqui (uma linha).",
    description="URL:",
    layout=widgets.Layout(width="90%", height="70px"),
)

infer_url_button = widgets.Button(description="Inferir (URL)", button_style="success")
out_url = widgets.Output()

def on_infer_url(_):
    if loaded_model is None:
        with out_url:
            clear_output()
            print("Carregue um modelo primeiro.")
        return

    url = (url_text.value or "").strip().splitlines()[0].strip()
    if not url:
        with out_url:
            clear_output()
            print("Informe uma URL.")
        return

    try:
        r = requests.get(url, timeout=15)
        r.raise_for_status()
        img = Image.open(io.BytesIO(r.content)).convert("RGB")
        preds = predict_topk(loaded_model, img, k=5)

        with out_url:
            clear_output()
            plt.figure(figsize=(4,4))
            plt.imshow(img); plt.axis("off"); plt.show()
            print("Top-5:")
            for name, p in preds:
                print(f"  {name:<12}  {p:.3f}")
    except Exception as e:
        with out_url:
            clear_output()
            print("Falha ao baixar/inferir:", repr(e))

infer_url_button.on_click(on_infer_url)

display(widgets.VBox([widgets.HTML("<b>2) Inferência por URL</b>"), url_text, infer_url_button, out_url]))

## 5) Webcam: inferência em tempo real com *top‑k*

Este bloco usa OpenCV. Em alguns ambientes, webcam pode não funcionar.  
Se falhar, use upload/URL (já cobre a transição “do modelo ao usuário final”).


In [ ]:
import threading
import time
import io
from PIL import Image

# --- UI ---
webcam_start   = widgets.Button(description="Iniciar",   button_style="warning")
webcam_stop    = widgets.Button(description="Parar",     button_style="danger")
webcam_restart = widgets.Button(description="Reiniciar", button_style="info")
fps_slider     = widgets.IntSlider(value=4, min=1, max=10, step=1, description="FPS:", continuous_update=False)

frame_widget = widgets.Image(
    format="jpeg",
    layout=widgets.Layout(width="480px"))   # mostra o frame atual
pred_widget  = widgets.HTML(value="")         # mostra o Top-k atual
status_widget = widgets.HTML(value="")        # status

panel = widgets.VBox([
    status_widget,
    frame_widget,
    pred_widget,
])

# --- Estado ---
stop_event = threading.Event()
state = {"thread": None, "cap": None}

def _release_cap():
    cap = state.get("cap")
    if cap is not None:
        try:
            cap.release()
        except Exception:
            pass
    state["cap"] = None

def _format_preds_html(preds):
    # tabela simples (fica “produto-like”)
    rows = []
    for name, p in preds:
        bar = "█" * int(p * 25)
        rows.append(f"<tr><td style='padding-right:12px;'><b>{name}</b></td><td>{p:.3f}</td><td style='padding-left:12px;font-family:monospace'>{bar}</td></tr>")
    return (
        "<div style='margin-top:8px'>"
        "<div><b>Top-5</b></div>"
        "<table style='border-collapse:collapse;margin-top:6px'>"
        + "".join(rows) +
        "</table></div>"
    )

def _update_panel(img_pil, preds):
    # Atualiza imagem
    buf = io.BytesIO()
    img_pil.save(buf, format="JPEG", quality=85)
    frame_widget.value = buf.getvalue()

    # Atualiza previsões
    pred_widget.value = _format_preds_html(preds)

def _webcam_loop():
    cap = state["cap"]
    if cap is None:
        return

    status_widget.value = "<b>Webcam rodando…</b> (use Parar/Reiniciar)"
    pred_widget.value = ""
    frame_widget.value = b""

    while not stop_event.is_set():
        ok, frame = cap.read()
        if not ok:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(rgb)

        preds = predict_topk(loaded_model, img, k=5)

        _update_panel(img, preds)

        fps = int(fps_slider.value)
        time.sleep(max(0.001, 1.0 / max(1, fps)))

    _release_cap()
    status_widget.value = "Webcam finalizada."

def _start_camera():
    if cv2 is None:
        status_widget.value = "OpenCV (cv2) não está disponível. Instale com: pip install opencv-python"
        return
    if loaded_model is None:
        status_widget.value = "Carregue um modelo primeiro."
        return

    t = state.get("thread")
    if t is not None and t.is_alive():
        status_widget.value = "Webcam já está rodando. Clique em Parar ou Reiniciar."
        return

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        status_widget.value = "Não foi possível abrir a webcam (id=0)."
        return

    state["cap"] = cap
    stop_event.clear()

    t = threading.Thread(target=_webcam_loop, daemon=True)
    state["thread"] = t
    t.start()

def start_webcam(_):
    _start_camera()

def stop_webcam(_):
    stop_event.set()
    _release_cap()

def restart_webcam(_):
    stop_event.set()
    _release_cap()
    time.sleep(0.2)
    stop_event.clear()
    _start_camera()

webcam_start.on_click(start_webcam)
webcam_stop.on_click(stop_webcam)
webcam_restart.on_click(restart_webcam)

display(widgets.VBox([
    widgets.HTML("<b>3) Inferência por webcam (opcional)</b>"),
    widgets.HBox([webcam_start, webcam_stop, webcam_restart, fps_slider]),
    panel
]))

## 6) Checklist de “produto”

- Qual **modelo** (arquivo `.pt`) está em uso?
- O pré‑processamento é **idêntico** ao treino?
- O top‑k faz sentido? Você consegue justificar erros por **domínio/distribuição**?
- Você consegue mostrar pelo menos:
  - um acerto claro
  - um erro claro com explicação técnica

Isso fecha: **treinar é engenharia; demonstrar é aplicação.**
